# Install the necessary packages

In [2]:
%pip install 'crewai[tools]' setuptools crewai


[notice] A new release of pip is available: 24.3.1 -> 25.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [3]:
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
from crewai import Agent, Task, Crew, Process
from crewai_tools import ScrapeWebsiteTool, SerperDevTool,PDFSearchTool

search_tool = SerperDevTool()
scrape_tool = ScrapeWebsiteTool()
# Healthcare Agents

patient_intake = Agent(
    role='Patient Intake Specialist',
    goal='Gather patient information and medical history, with human verification.',
    backstory="You are a meticulous intake specialist, ensuring accurate patient records. You understand that human verification is crucial for sensitive data.",
    verbose=True,
    allow_delegation=True
)
diagnostic_expert = Agent(
    role='Diagnostic Expert',
    goal='Analyze patient symptoms and medical history to suggest potential diagnoses, with human verification and external search.',
    backstory="You are a seasoned diagnostician with extensive medical knowledge. You understand that in complex cases human validation is needed. You also utilize external search and website scraping to find the latest advancements.",
    verbose=True,
    allow_delegation=True,
    tools=[search_tool, scrape_tool]  # Add search and scrape tools
)

treatment_planner = Agent(
    role='Treatment Planner',
    goal='Develop personalized treatment plans based on diagnoses, with human approval and external search.',
    backstory="You are a skilled treatment planner, focusing on effective and patient-centered care. You will ask for human approval before finalizing the treatment plan. You also utilize external search and website scraping to find the latest advancements.",
    verbose=True,
    allow_delegation=True,
    tools=[search_tool, scrape_tool]  # Add search and scrape tools
)

pharmacy_advisor = Agent(
    role='Pharmacy Advisor',
    goal='Provide information on medications, dosages, and potential interactions.',
    backstory="You are a knowledgeable pharmacy advisor, ensuring safe medication practices.",
    verbose=True,
    allow_delegation=False
)

wellness_coach = Agent(
    role='Wellness Coach',
    goal='Advise on lifestyle changes and wellness strategies to support patient health.',
    backstory="You are a compassionate wellness coach, promoting holistic health.",
    verbose=True,
    allow_delegation=False
)

medical_record_keeper = Agent(
    role='Medical Record Keeper',
    goal='Maintain and update patient medical records with accuracy and confidentiality.',
    backstory="You are a meticulous medical record keeper, ensuring data integrity.",
    verbose=True,
    allow_delegation=False
)

# Healthcare Tasks

intake_task = Task(
    description="Gather patient information, including symptoms, medical history, and current medications from the given info {input}.Do not make any assumptions stick to the data given by the user. After gathering the information, present it for human verification. If the human input is 'OK' continue, if it is not, correct the information.",
    agent=patient_intake,
    expected_output="Detailed, human-verified patient information and medical history.",
    human_input=True
)

diagnostic_task = Task(
    description="Analyze the patient's symptoms and medical history to suggest potential diagnoses. Provide a list of possible conditions and relevant tests. If any disease is rare, or the symptoms are complex, ask for a human verification by presenting the gathered information, and potential diagnoses.",
    agent=diagnostic_expert,
    expected_output="A list of potential diagnoses and recommended tests, verified by a human when needed.",
    human_input=True #This will be triggered if the agent determines it is needed.
)

treatment_task = Task(
    description="Based on the diagnostic information, develop a personalized treatment plan, including medications, therapies, and follow-up appointments. Provide the medications needed to the pharmacy advisor. Once you have a treatment plan, present it for human approval, and wait for human input. Only continue if the human input is 'Approved'.",
    agent=treatment_planner,
    expected_output="A comprehensive, human-approved treatment plan.",
    human_input=True
)

pharmacy_task = Task(
    description="Provide information on the medications prescribed in the treatment plan, including dosages, potential side effects, and drug interactions. Send the safe medication usage to the treatment planner and wellness coach.",
    agent=pharmacy_advisor,
    expected_output="Medication information and safety guidelines."
)

wellness_task = Task(
    description="Based on the treatment plan and patient history, provide lifestyle and wellness advice to support the patient's recovery and overall health. Incorporate the safe medication usage from the pharmacy advisor.",
    agent=wellness_coach,
    expected_output="Personalized wellness and lifestyle recommendations."
)

record_keeping_task = Task(
    description="Update the patient's medical records with all information gathered and generated during the process, ensuring accuracy and confidentiality. Summarize all steps for the patient intake.",
    agent=medical_record_keeper,
    expected_output="Updated medical records and a summary of the patient's healthcare journey."
)

# Healthcare Crew

healthcare_crew = Crew(
    agents=[patient_intake, diagnostic_expert, treatment_planner, pharmacy_advisor, wellness_coach, medical_record_keeper],
    tasks=[intake_task, diagnostic_task, treatment_task, pharmacy_task, wellness_task, record_keeping_task],
    verbose=True,
    process=Process.sequential,# Tasks are executed sequentially
    planning=True,
    planning_llm='gpt-4o'  
)
# Kickoff the healthcare crew
patient_data = "Patient: John Doe, 45 years old. Symptoms: Persistent cough, fever, fatigue. Medical history: Asthma, allergies. Current medications: Albuterol inhaler."

result = healthcare_crew.kickoff(inputs={"input": patient_data})

print("\nHealthcare Process Result:")
print(result)

2025-03-20 17:41:39,682 - 8306786880 - __init__.py-__init__:537 - WARNING: Overriding of current TracerProvider is not allowed


 
[2025-03-20 17:41:39][INFO]: Planning the crew execution
# Agent: Patient Intake Specialist
## Task: Gather patient information, including symptoms, medical history, and current medications from the given info Patient: John Doe, 45 years old. Symptoms: Persistent cough, fever, fatigue. Medical history: Asthma, allergies. Current medications: Albuterol inhaler..Do not make any assumptions stick to the data given by the user. After gathering the information, present it for human verification. If the human input is 'OK' continue, if it is not, correct the information.1. The Patient Intake Specialist will review the provided patient information: John Doe, 45 years old, symptoms of persistent cough, fever, fatigue. Medical history includes asthma and allergies. Current medication is an Albuterol inhaler.
2. Ensure that the information does not have additional missing data or discrepancies. 
3. Present the information to the human verifier for confirmation.
4. If the human input is 'OK', f